# PID-RL API Server - Hackathon
Model: Fenryr/qwen2.5-1.5B-pid-v1## Instructions1. Runtime > Change runtime type > GPU (T4)2. Run cells in order 1-83. Enter HF token when asked

## Cell 1: Install Dependencies

In [ ]:
# Install required packages

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install transformers unsloth peft accelerate

In [ ]:
!pip install fastapi uvicorn pydantic cloudflared yfinance

In [ ]:
print('Dependencies installed')

## Cell 2: Enter HF Token

In [ ]:
# Enter your HF token

In [ ]:
from getpass import getpass

In [ ]:
HF_TOKEN = getpass('Enter HF token: ')

In [ ]:
import os

In [ ]:
os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
print('Token set')

## Cell 3: Clone Repo

In [ ]:
import subprocess

In [ ]:
import sys

In [ ]:
import os

In [ ]:
REPO_DIR = '/content/qwen-rl-shanghai'

In [ ]:
if not os.path.isdir(REPO_DIR):

In [ ]:
    subprocess.run(['git', 'clone', 'https://github.com/Grinta-Protocol/qwen-rl-shanghai.git', REPO_DIR], check=True)

In [ ]:
sys.path.insert(0, f'{REPO_DIR}/pid_rl')

In [ ]:
os.chdir(f'{REPO_DIR}/pid_rl')

In [ ]:
print('Repo cloned')

## Cell 4: Load Model

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
import torch

In [ ]:
from peft import PeftModel

In [ ]:
LORA_PATH = 'Fenryr/qwen2.5-1.5B-pid-v1'

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'

In [ ]:
MAX_SEQ_LEN = 2048

In [ ]:
print('Loading base model...')

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(

In [ ]:
    model_name=BASE_MODEL,

In [ ]:
    max_seq_length=MAX_SEQ_LEN,

In [ ]:
    load_in_4bit=False,

In [ ]:
    fast_inference=False

In [ ]:
)

In [ ]:
print('Loading LoRA adapter...')

In [ ]:
model = FastLanguageModel.get_peft_model(

In [ ]:
    model,

In [ ]:
    r=8,

In [ ]:
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],

In [ ]:
    lora_alpha=8

In [ ]:
)

In [ ]:
model = PeftModel.from_pretrained(model, LORA_PATH)

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
print('Model loaded')

## Cell 5: Inference Wrapper

In [ ]:
from prompt import build_prompt, parse_output

In [ ]:
from types import SimpleNamespace

In [ ]:
import numpy as np

In [ ]:
class TrainedModelPolicy:

In [ ]:
    name = 'trained'

In [ ]:
    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.3):

In [ ]:
        self.model = model

In [ ]:
        self.tokenizer = tokenizer

In [ ]:
        self.max_new_tokens = max_new_tokens

In [ ]:
        self.temperature = temperature

In [ ]:
    def decide(self, state, btc_history, deviation_history):

In [ ]:
        sys_p, user_p = build_prompt(state=state, btc_history=btc_history, deviation_history=deviation_history)

In [ ]:
        messages = [{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': user_p}]

In [ ]:
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
        inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)

In [ ]:
        with torch.no_grad():

In [ ]:
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, temperature=self.temperature, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)

In [ ]:
        gen = self.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
        return gen

In [ ]:
trained_policy = TrainedModelPolicy(model, tokenizer)

In [ ]:
print('Inference ready')

## Cell 6: FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException

In [ ]:
from pydantic import BaseModel

In [ ]:
import threading

In [ ]:
import uvicorn

In [ ]:
app = FastAPI(title='PID-RL API')

In [ ]:
class InferenceRequest(BaseModel):

In [ ]:
    state: dict

In [ ]:
    btc_history: list

In [ ]:
    deviation_history: list

In [ ]:
class InferenceResponse(BaseModel):

In [ ]:
    kp: float

In [ ]:
    ki: float

In [ ]:
    raw_output: str

In [ ]:
    success: bool

In [ ]:
@app.get('/')

In [ ]:
def root():

In [ ]:
    return {'status': 'ok', 'model': 'Fenryr/qwen2.5-1.5B-pid-v1'}

In [ ]:
@app.get('/health')

In [ ]:
def health():

In [ ]:
    return {'status': 'healthy'}

In [ ]:
@app.post('/predict', response_model=InferenceResponse)

In [ ]:
def predict(req: InferenceRequest):

In [ ]:
    try:

In [ ]:
        state = SimpleNamespace(**req.state)

In [ ]:
        output = trained_policy.decide(state=state, btc_history=req.btc_history, deviation_history=req.deviation_history)

In [ ]:
        try:

In [ ]:
            parsed = parse_output(output)

In [ ]:
        except:

In [ ]:
            parsed = {}

In [ ]:
        return InferenceResponse(kp=parsed.get('kp', state.kp), ki=parsed.get('ki', state.ki), raw_output=output[:500], success=True)

In [ ]:
    except Exception as e:

In [ ]:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
def run_server():

In [ ]:
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

In [ ]:
threading.Thread(target=run_server, daemon=True).start()

In [ ]:
import time

In [ ]:
time.sleep(2)

In [ ]:
print('Server ready on port 8000')

## Cell 7: Cloudflare Tunnel

In [ ]:
!pip install cloudflared

In [ ]:
print('Cloudflared installed')

In [ ]:
import subprocess

In [ ]:
import time

In [ ]:
subprocess.Popen(['nohup', 'cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
time.sleep(10)

In [ ]:
log = open('/tmp/tunnel.log').read()

In [ ]:
if 'trycloudflare.com' in log:

In [ ]:
    for line in log.splitlines():

In [ ]:
        if 'https://' in line and 'trycloudflare.com' in line:

In [ ]:
            url = line.strip().split()[0]

In [ ]:
            break

In [ ]:
    print('='*60)

In [ ]:
    print('TUNNEL READY')

In [ ]:
    print('='*60)

In [ ]:
    print('URL:', url)

In [ ]:
    print('POST', url + '/predict')

In [ ]:
else:

In [ ]:
    print('Waiting for tunnel...')

In [ ]:
    print(log)

## Cell 8: Test

In [ ]:
import requests

In [ ]:
import json

In [ ]:
test = {'state': {'deviation_pct': 2.5, 'kp': 1.0, 'ki': 0.01, 'integral_error': 5.0}, 'btc_history': [42000, 42100, 41900, 41800, 41750], 'deviation_history': [1.2, 1.5, 1.8, 2.0, 2.2]}

In [ ]:
try:

In [ ]:
    r = requests.post('http://localhost:8000/predict', json=test, timeout=60)

In [ ]:
    print('OK')

In [ ]:
    print(json.dumps(r.json(), indent=2))

In [ ]:
except Exception as e:

In [ ]:
    print('Error:', e)